# Notebook 03 — MCMC: From Random Walks to Hamiltonian Dynamics

## Background

Every Bayesian analysis reduces to the same problem: compute the posterior distribution `P(θ | data)`. In the last two notebooks we got lucky — our models had conjugate priors, so the posterior had a closed-form solution. In real problems, that almost never happens.

When you're modeling a hierarchical experiment with 50 parameters, or fitting a GP with a non-standard kernel, or doing Bayesian neural network inference, there is no formula for the posterior. It lives in a high-dimensional space and all we can do is *sample* from it. That's what Markov Chain Monte Carlo (MCMC) does: it builds a Markov chain whose stationary distribution is exactly your posterior, then runs the chain long enough to collect representative samples.

## What You'll Learn

- Why MCMC works: detailed balance, stationary distributions, ergodicity
- Metropolis-Hastings: the classic random-walk algorithm (conceptual foundation)
- Gibbs sampling: why it was popular, why it's largely superseded
- Hamiltonian Monte Carlo: using gradient information to make large, informed proposals
- NUTS: the No-U-Turn Sampler — HMC without manual tuning
- PyMC in practice: `pm.sample()`, `az.plot_trace()`, `az.summary()`, R-hat, ESS, divergences

## Why This Matters

Until ~2011, practitioners either used conjugate models (limited flexibility) or ran Gibbs samplers that required deriving full conditionals by hand — a significant mathematical burden for each new model. HMC, and then NUTS (Hoffman & Gelman 2014), changed the game by making it possible to sample from *almost any* posterior automatically, just by specifying the log-probability and letting automatic differentiation handle the gradients. Stan and PyMC are both built on this foundation. Understanding MCMC at this level lets you diagnose failures (divergences, low ESS, high R-hat) and fix them — skills that are invisible to practitioners who treat `pm.sample()` as a black box.

## Part 1 — Why Random Sampling is the Answer

Suppose your posterior is a distribution over parameters `θ = (θ₁, θ₂, ..., θₖ)`. You want to compute things like:

- `E[θ₁ | data]` — posterior mean
- `P(θ₁ > 0 | data)` — probability of a directional effect
- Credible intervals, predictions, etc.

All of these are integrals: `E[f(θ) | data] = ∫ f(θ) · P(θ|data) dθ`.

In low dimensions (1-2 parameters), you can compute these numerically on a grid. In 50 dimensions, a grid with 10 points per dimension has 10⁵⁰ cells — physically impossible. The curse of dimensionality makes grid methods useless for real models.

**Monte Carlo integration** gives you an escape hatch: if you have samples `θ⁽¹⁾, θ⁽²⁾, ..., θ⁽ᴺ⁾` drawn from `P(θ|data)`, then:

```
E[f(θ) | data] ≈ (1/N) Σ f(θ⁽ⁱ⁾)
```

This converges at `O(1/√N)` regardless of dimensionality. The challenge: **how do you draw samples from a distribution you can only evaluate up to a normalizing constant?**

You know `P(θ|data) ∝ P(data|θ) · P(θ)` — you can compute the numerator at any point, but the denominator `P(data)` requires the very integral you're trying to avoid. MCMC sidesteps this entirely.

## Part 2 — Metropolis-Hastings: The Random Walk Approach

*(Conceptual foundation — we use PyMC for all actual sampling)*

The Metropolis-Hastings algorithm (Metropolis et al. 1953; Hastings 1970) is elegant in its simplicity. Start at any point `θ_current`. At each step:

1. **Propose**: draw a candidate `θ_proposed ~ Q(θ | θ_current)` — typically a Gaussian centered at `θ_current`
2. **Accept/reject** with probability:

```
α = min(1,  P(θ_proposed|data) · Q(θ_current|θ_proposed)
            ─────────────────────────────────────────────)
            P(θ_current|data)  · Q(θ_proposed|θ_current)
```

The normalizing constant `P(data)` cancels in the ratio — that's the key insight.

**Why this works**: the acceptance rule enforces *detailed balance* — the probability flux from A→B equals B→A. Any chain satisfying detailed balance has the target distribution as its stationary distribution.

**The problem with random walks**: MH with a Gaussian proposal does a *random walk* through parameter space. In high dimensions this is catastrophically inefficient:
- Small step size → high acceptance but slow exploration (highly correlated samples)
- Large step size → proposals land in low-probability regions → very low acceptance
- The optimal step size scales as `O(d^{-1/2})` in `d` dimensions
- For a 50-parameter model, you might need millions of proposals for 1,000 effective samples

This is why random-walk MH, while theoretically valid, is impractical for real Bayesian models.

## Part 3 — Gibbs Sampling: When You Have Full Conditionals

*(Historical context — superseded by HMC for most practical work)*

Gibbs sampling (Geman & Geman 1984; popularized by Gelfand & Smith 1990) avoids proposal tuning by cycling through each parameter in turn, sampling from the *full conditional* — the distribution of `θᵢ` given all other parameters and data.

```python
# Conceptual Gibbs sampler
for iteration in range(n_samples):
    θ₁ = sample_from_P(θ₁ | θ₂, θ₃, ..., θₖ, data)
    θ₂ = sample_from_P(θ₂ | θ₁, θ₃, ..., θₖ, data)
    ...
    θₖ = sample_from_P(θₖ | θ₁, θ₂, ..., θₖ₋₁, data)
```

**Why it was popular**: For conjugate models, each full conditional is a known distribution that's easy to sample from. BUGS and JAGS used Gibbs sampling to make Bayesian inference accessible for decades.

**Why it's superseded**: Two serious limitations:

1. **Strong correlations → slow mixing**: If `θ₁` and `θ₂` are highly correlated in the posterior (routine in hierarchical models), Gibbs takes tiny steps along the narrow posterior diagonal. Thousands of iterations may explore the same narrow region. HMC's global proposals cut across the entire posterior in one step.

2. **Requires analytical full conditionals**: Non-conjugate likelihoods (neural networks, complex observation models) break this requirement.

HMC solves both: it makes long-range proposals by following gradient-guided trajectories, and only needs `∇ log P(θ|data)` — available for almost any differentiable model via autograd.

## Part 4 — Hamiltonian Monte Carlo: Physics-Inspired Exploration

HMC (Duane et al. 1987; Neal 2011) reframes sampling as a physics simulation. Introduce auxiliary *momentum* variables `p` for each parameter `θ`:

```
P(θ, p) ∝ exp(-U(θ) - K(p))
```

Where:
- `U(θ) = -log P(θ|data)` — **potential energy** (low where posterior is high)
- `K(p) = p²/(2M)` — **kinetic energy** (Gaussian on momentum)
- `H(θ, p) = U(θ) + K(p)` — total Hamiltonian (conserved energy)

**The algorithm**:
1. Sample fresh momentum: `p ~ N(0, M)`
2. Simulate Hamiltonian dynamics for `L` leapfrog steps (step size `ε`):
   ```
   p ← p - (ε/2) ∇U(θ)   # half-step momentum
   θ ← θ + ε M⁻¹ p        # full-step position
   p ← p - (ε/2) ∇U(θ)   # half-step momentum
   ```
3. Accept/reject `(θ', p')` with probability `min(1, exp(H_current - H_proposed))`

**Why this is so much better**:

Hamiltonian dynamics are *volume-preserving* (Liouville's theorem) and energy-conserving (in continuous time). The leapfrog integrator preserves these properties approximately, so `(θ', p')` has nearly the same total energy as the start — almost every proposal is accepted. Meanwhile, `θ'` has traveled far from `θ` by following the gradient contours of the posterior.

Think of it as a marble rolling on a bowl: the gradient guides the marble to explore the valley efficiently, rather than a drunk stumbling randomly.

**The remaining problem**: you still need to tune `L` (leapfrog steps) and `ε` (step size). Too few steps → correlated samples. Too many → trajectory U-turns and wasted computation.

## Part 5 — NUTS: Automatic HMC

The No-U-Turn Sampler (Hoffman & Gelman 2014) solves the leapfrog tuning problem. NUTS extends the trajectory dynamically, stopping when it starts turning back:

```
Stop when: (θ' - θ) · p' < 0
```

This detects the "U-turn" — when momentum points back toward where you started — and stops just before, so you never retrace old ground. Trajectory length is selected automatically per sample.

NUTS also uses **dual averaging** for adaptive step size during warmup, targeting a specific acceptance rate (default 0.8 in PyMC). After warmup, adaptation stops and the step size is fixed.

**Result**: `pm.sample()` — specify the model, NUTS handles the rest. This is why PyMC and Stan made Bayesian inference accessible to practitioners without deep MCMC expertise.

**Divergences**: The key failure mode. A *divergence* occurs when the leapfrog integrator makes a numerical error — the trajectory flies off into low-probability space. This happens in regions of high posterior curvature ("funnel" geometries, common in hierarchical models). Even a few divergences indicate potential bias. The fix is reparameterization — covered in notebook 05.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# Visualize the MH random walk efficiency problem on a correlated 2D posterior
np.random.seed(42)

def log_target(theta, rho=0.95):
    """Log of correlated 2D Gaussian — representative of hierarchical posteriors."""
    cov_inv = np.array([[1, -rho], [-rho, 1]]) / (1 - rho**2)
    return -0.5 * theta @ cov_inv @ theta

def metropolis_hastings(n_samples, step_size, rho=0.95, seed=42):
    """Gaussian random-walk MH sampler."""
    np.random.seed(seed)
    samples = np.zeros((n_samples, 2))
    current = np.zeros(2)
    n_accepted = 0
    for i in range(n_samples):
        proposed = current + np.random.normal(0, step_size, size=2)
        log_alpha = log_target(proposed, rho) - log_target(current, rho)
        if np.log(np.random.uniform()) < log_alpha:
            current = proposed
            n_accepted += 1
        samples[i] = current
    return samples, n_accepted / n_samples

def effective_sample_size(x):
    """Estimate ESS via autocorrelation sum."""
    n = len(x)
    x_c = x - x.mean()
    acf = np.correlate(x_c, x_c, mode='full')[n-1:] / (x_c.var() * n)
    cutoff = np.where(acf < 0)[0]
    cutoff = cutoff[0] if len(cutoff) else len(acf)
    tau = 1 + 2 * acf[1:cutoff].sum()
    return n / max(tau, 1)

n_samples = 2000
samples_small, acc_small = metropolis_hastings(n_samples, 0.1)
samples_large, acc_large = metropolis_hastings(n_samples, 5.0)
samples_tuned, acc_tuned = metropolis_hastings(n_samples, 0.5)

print('Metropolis-Hastings on highly correlated posterior (rho=0.95)')
print(f'{"Step size":>12} {"Accept rate":>13} {"ESS":>8} {"ESS/n":>8}')
print('-' * 46)
for label, samples, acc in [
    ('0.1 (small)', samples_small, acc_small),
    ('5.0 (large)', samples_large, acc_large),
    ('0.5 (tuned)', samples_tuned, acc_tuned),
]:
    ess = effective_sample_size(samples[:, 0])
    print(f'{label:>12} {acc:>13.2%} {ess:>8.0f} {ess/n_samples:>8.3f}')

In [ ]:
# Visualize the sampling paths on the correlated posterior
x_range = np.linspace(-4, 4, 200)
X, Y = np.meshgrid(x_range, x_range)
rho = 0.95
Z = np.exp(-0.5 * (X**2 - 2*rho*X*Y + Y**2) / (1 - rho**2))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
show_n = 300

configs = [
    (samples_small, acc_small, 'Step=0.1 (too small)'),
    (samples_large, acc_large, 'Step=5.0 (too large)'),
    (samples_tuned, acc_tuned, 'Step=0.5 (tuned)'),
]

for ax, (samples, acc, title) in zip(axes, configs):
    ess = effective_sample_size(samples[:, 0])
    ax.contour(X, Y, Z, levels=5, colors='gray', alpha=0.4, linewidths=1)
    ax.plot(samples[:show_n, 0], samples[:show_n, 1], 'b-', alpha=0.2, linewidth=0.5)
    sc = ax.scatter(samples[:show_n, 0], samples[:show_n, 1],
                    c=np.arange(show_n), cmap='viridis', s=6, alpha=0.7)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_title(f'{title}\nAccept: {acc:.1%}  ESS: {ess:.0f}/{show_n}', fontsize=10)
    ax.set_xlabel('theta_1'); ax.set_ylabel('theta_2')

plt.suptitle('Metropolis-Hastings on Correlated Posterior (rho=0.95)\n'
             'Color = iteration order (purple -> yellow = early -> late)', fontsize=11)
plt.tight_layout()
plt.show()
print('Random-walk MH struggles with correlated geometry regardless of step size.')
print('HMC follows gradient contours and escapes this problem.')

## Part 6 — PyMC and NUTS in Practice

Now we use PyMC to sample from a real posterior. We'll fit a logistic regression — a non-conjugate model — to show that NUTS handles it as easily as anything else.

**Scenario**: Analyzing the effect of study hours on exam pass rates. Data from 50 students.

In [ ]:
import pymc as pm
import arviz as az

print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

np.random.seed(42)
n_students = 50
true_intercept, true_slope = -3.0, 1.2

hours = np.random.uniform(0, 5, n_students)
log_odds_true = true_intercept + true_slope * hours
prob_pass_true = 1 / (1 + np.exp(-log_odds_true))
passed = np.random.binomial(1, prob_pass_true)

print(f'\nDataset: {n_students} students')
print(f'Hours studied: {hours.min():.1f} -- {hours.max():.1f}')
print(f'Pass rate: {passed.mean():.1%}')
print(f'True intercept: {true_intercept}, True slope: {true_slope}')

In [ ]:
with pm.Model() as logistic_model:
    # Weakly informative priors -- standard for logistic regression
    intercept = pm.Normal('intercept', mu=0, sigma=5)
    slope = pm.Normal('slope', mu=0, sigma=5)
    
    # Likelihood via logit parameterization
    y = pm.Bernoulli('y', logit_p=intercept + slope * hours, observed=passed)
    
    # NUTS is the default sampler in PyMC
    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,  # higher -> smaller step size, fewer divergences
        random_seed=42,
        progressbar=True
    )

In [ ]:
# Convergence diagnostics -- run these after every pm.sample() call
summary = az.summary(trace, var_names=['intercept', 'slope'])
print('Posterior Summary:')
print(summary.to_string())
print()

n_divergent = int(trace.sample_stats['diverging'].values.sum())
print(f'Divergences: {n_divergent}')
print(f'  Status: {"No issues" if n_divergent == 0 else "WARNING -- investigate"}')

print(f'\nTrue vs recovered:')
print(f'  intercept: true={true_intercept}, posterior_mean={summary.loc["intercept", "mean"]:.2f}')
print(f'  slope:     true={true_slope}, posterior_mean={summary.loc["slope", "mean"]:.2f}')

In [ ]:
# Trace plots -- the standard visual convergence check
az.plot_trace(trace, var_names=['intercept', 'slope'], figsize=(12, 5))
plt.suptitle('Trace Plots: Left=KDE, Right=Trace over iterations\n'
             'All 4 chains should overlap; traces should look stationary', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Posterior distributions with HDI
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, var, true_val in zip(axes,
                              ['intercept', 'slope'],
                              [true_intercept, true_slope]):
    az.plot_posterior(trace, var_names=[var], hdi_prob=0.94,
                      point_estimate='mean', ax=ax)
    ax.axvline(true_val, color='red', linestyle='--', label=f'True: {true_val}')
    ax.legend(fontsize=9)
plt.suptitle('Posterior Distributions -- 94% Highest Density Interval', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Posterior predictive: pass probability curve with uncertainty
intercept_samp = trace.posterior['intercept'].values.flatten()
slope_samp = trace.posterior['slope'].values.flatten()

hours_range = np.linspace(0, 5, 100)
log_odds_mat = intercept_samp[:, None] + slope_samp[:, None] * hours_range[None, :]
prob_mat = 1 / (1 + np.exp(-log_odds_mat))
true_curve = 1 / (1 + np.exp(-(true_intercept + true_slope * hours_range)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(hours_range,
                np.percentile(prob_mat, 3, axis=0),
                np.percentile(prob_mat, 97, axis=0),
                alpha=0.2, color='steelblue', label='94% CI')
ax.fill_between(hours_range,
                np.percentile(prob_mat, 25, axis=0),
                np.percentile(prob_mat, 75, axis=0),
                alpha=0.4, color='steelblue', label='50% CI')
ax.plot(hours_range, prob_mat.mean(axis=0), 'steelblue', lw=2, label='Posterior mean')
ax.plot(hours_range, true_curve, 'r--', lw=2, label='True probability')
ax.scatter(hours[passed==1], np.ones(passed.sum()) * 1.02, color='green', alpha=0.5, s=25, label='Passed')
ax.scatter(hours[passed==0], np.zeros((~passed.astype(bool)).sum()) - 0.02, color='red', alpha=0.5, s=25, marker='x', label='Failed')
ax.set_xlabel('Hours Studied'); ax.set_ylabel('P(Pass)')
ax.set_title('Posterior Predictive: Pass Probability vs. Study Hours')
ax.legend(); ax.set_ylim(-0.1, 1.15)
plt.tight_layout()
plt.show()

for h in [2.0, 3.0, 4.0]:
    lo = intercept_samp + slope_samp * h
    p = 1 / (1 + np.exp(-lo))
    lo_ci, hi_ci = np.percentile(p, [3, 97])
    print(f'After {h:.0f}h: P(pass) = {p.mean():.1%} (94% CI: {lo_ci:.1%} -- {hi_ci:.1%})')

## Part 7 — Diagnosing MCMC Problems

NUTS is robust, but it can fail. Reading the diagnostic signals is an essential skill.

In [ ]:
# Simulate what good vs. badly-mixed chains look like
np.random.seed(42)
n = 1000

good_chains = np.array([np.random.normal(0, 1, n) for _ in range(4)])
bad_chains = np.array([
    np.random.normal(-3, 0.3, n),
    np.random.normal(-3, 0.3, n),
    np.random.normal(+3, 0.3, n),
    np.random.normal(+3, 0.3, n),
])

def rhat(chains):
    """Gelman-Rubin R-hat: compares between-chain to within-chain variance.
    R-hat near 1.0 means chains sampled the same distribution.
    R-hat > 1.01 is a warning; > 1.05 is a failure.
    """
    m, n = chains.shape
    chain_means = chains.mean(axis=1)
    chain_vars = chains.var(axis=1, ddof=1)
    B = n * chain_means.var(ddof=1)   # between-chain variance
    W = chain_vars.mean()              # within-chain variance
    var_hat = (n - 1) / n * W + B / n # pooled estimate
    return np.sqrt(var_hat / W)

print('Convergence Diagnostics')
print(f'{"":15} {"R-hat":>8}')
print('-' * 26)
print(f'{"Good chains":15} {rhat(good_chains):>8.4f}  <- target: < 1.01')
print(f'{"Bad chains":15} {rhat(bad_chains):>8.4f}  <- FAIL: chains in different modes')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
colors = ['steelblue', 'darkorange', 'green', 'red']

for col, (chains, label, r) in enumerate([
    (good_chains, 'Well-mixed', rhat(good_chains)),
    (bad_chains, 'Poorly-mixed (stuck in modes)', rhat(bad_chains)),
]):
    ax_trace = axes[0, col]
    for i, chain in enumerate(chains):
        ax_trace.plot(chain[:200], alpha=0.7, lw=0.8, color=colors[i], label=f'Chain {i+1}')
    ax_trace.set_title(f'{label} -- R-hat = {r:.4f}', fontsize=10)
    ax_trace.set_xlabel('Iteration'); ax_trace.set_ylabel('theta')
    ax_trace.legend(fontsize=8, loc='upper right')

    ax_kde = axes[1, col]
    xr = np.linspace(chains.min() - 1, chains.max() + 1, 300)
    for i, chain in enumerate(chains):
        kde = gaussian_kde(chain)
        ax_kde.plot(xr, kde(xr), alpha=0.8, color=colors[i], label=f'Chain {i+1}')
    ax_kde.set_title('Marginal distributions per chain', fontsize=10)
    ax_kde.set_xlabel('theta'); ax_kde.set_ylabel('Density')
    ax_kde.legend(fontsize=8)

plt.suptitle('MCMC Convergence: Good vs. Poorly-Mixed Chains', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Neal's Funnel -- the archetypal hard posterior geometry
# Common in hierarchical models; causes divergences in centered parameterization

np.random.seed(42)
n_pts = 5000
v = np.random.normal(0, 3, n_pts)          # group-level variance (log scale)
x_centered = np.random.normal(0, np.exp(v / 2))  # individual-level effect
x_raw = np.random.normal(0, 1, n_pts)      # non-centered version

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(x_centered, v, alpha=0.08, s=4, color='steelblue')
ax.set_xlim(-30, 30); ax.set_ylim(-9, 9)
ax.set_xlabel('x (individual effect)'); ax.set_ylabel('v (log group scale)')
ax.set_title("Neal's Funnel -- Centered Parameterization\n"
             "Narrow bottom requires tiny step size -> divergences")

ax = axes[1]
ax.scatter(x_raw, v, alpha=0.08, s=4, color='seagreen')
ax.set_xlim(-4, 4); ax.set_ylim(-9, 9)
ax.set_xlabel('x_raw (N(0,1))'); ax.set_ylabel('v (log group scale)')
ax.set_title('Non-Centered Parameterization\n'
             'Flat rectangle -> HMC works well, no divergences')

plt.suptitle("Funnel Geometry: The Key Challenge in Hierarchical Models\n"
             "Fix: x = x_raw * exp(v/2) instead of x ~ N(0, exp(v/2)) directly",
             fontsize=11)
plt.tight_layout()
plt.show()

print('Centered:     x ~ Normal(0, exp(v/2))')
print('Non-centered: x_raw ~ Normal(0, 1);  x = x_raw * exp(v/2)')
print()
print('Both represent the same model. The non-centered version has nearly')
print('independent x_raw and v -- flat geometry that HMC navigates easily.')
print('Divergences disappear. Covered in depth in notebook 05.')

In [ ]:
def mcmc_checklist(trace, var_names=None):
    """Post-sampling diagnostic checklist. Returns pass/warn/fail for each check."""
    summary = az.summary(trace, var_names=var_names)
    n_div = int(trace.sample_stats['diverging'].values.sum())
    max_rhat = float(summary['r_hat'].max())
    min_ess_bulk = float(summary['ess_bulk'].min())
    min_ess_tail = float(summary['ess_tail'].min())

    checks = {
        'divergences':  (n_div,         n_div == 0,      n_div < 10),
        'r_hat_max':    (round(max_rhat, 4), max_rhat < 1.01, max_rhat < 1.05),
        'ess_bulk_min': (int(min_ess_bulk), min_ess_bulk >= 400, min_ess_bulk >= 100),
        'ess_tail_min': (int(min_ess_tail), min_ess_tail >= 400, min_ess_tail >= 100),
    }

    print('MCMC Diagnostic Checklist')
    print('=' * 45)
    all_pass = True
    for name, (val, is_pass, is_warn) in checks.items():
        if is_pass:
            status = '[OK] '
        elif is_warn:
            status = '[!]  '
            all_pass = False
        else:
            status = '[FAIL]'
            all_pass = False
        print(f'  {status} {name:20} = {val}')
    print()
    print('Overall:', 'All checks passed.' if all_pass else 'Some checks need attention.')

mcmc_checklist(trace, var_names=['intercept', 'slope'])

In [ ]:
# Forest plot + pair plot -- additional standard ArviZ diagnostics
az.plot_forest(trace, var_names=['intercept', 'slope'],
               combined=True, hdi_prob=0.94, figsize=(8, 3))
plt.title('Forest Plot -- Posterior Means and 94% HDI')
plt.tight_layout()
plt.show()

az.plot_pair(trace, var_names=['intercept', 'slope'],
             kind='hexbin', marginals=True, figsize=(6, 6))
plt.suptitle('Joint Posterior: intercept vs. slope', y=1.01)
plt.tight_layout()
plt.show()

print('The negative intercept-slope correlation is structural:')
print('for fixed pass rates, a higher intercept requires a lower slope to fit the same data.')

## Summary

| Algorithm | Proposal | Requires | Scales to high-d? | Status |
|-----------|----------|----------|-------------------|--------|
| Random-walk MH | Gaussian step | `log P(θ\|data)` | No | Superseded |
| Gibbs | Full conditionals | Closed-form conditionals | No | Legacy |
| HMC | Leapfrog trajectory | `grad log P(θ\|data)` | Yes | Needs tuning |
| NUTS | Auto-stopped HMC | `grad log P(θ\|data)` | Yes | **Default** |

**Diagnostic quick reference**:
- **R-hat < 1.01** — chains converged to same distribution
- **ESS > 400** — enough independent samples for reliable estimates
- **0 divergences** — no numerical instability in posterior geometry
- Divergences → try `target_accept=0.95`, or non-centered reparameterization (notebook 05)

**Next**: Notebook 04 — PyMC Fundamentals. Deeper coverage of the model-building API: `pm.Data`, prior predictive checks, deterministics, multiple likelihoods, and the full workflow for real data analysis.